# ReefScan-Edge — GPU benchmark runner (Rungs 1–2)

Runs the inference-optimization harness on a Colab GPU. Self-contained: clones the public
repo and pulls the trained DINOv2-B checkpoint + the fixed 1,565-image test set from HF
(both public, no token needed).

**Before you start:** `Runtime -> Change runtime type -> L4 GPU` (or T4), then `Run all`.

Each rung registers a `(runtime, precision, device, batch)` variant into the harness, which
times with warmup + `cuda.synchronize()`, computes macro-F1 on the same test set, and appends
to `edge/results.csv` + `edge/RESULTS.md`. Re-running is **idempotent** — a rerun replaces a
variant's rows, it does not duplicate them, so `Run all` as many times as you like.

## 1. Confirm the GPU

In [ ]:
!nvidia-smi -L

## 2. Clone repo + pinned GPU deps
torch/vision come from the cu121 index so the GPU build matches the pinned 2.4.1.
(torch.compile's Inductor/Triton backend ships inside the torch CUDA wheel — no extra install.)

In [ ]:
!git clone https://github.com/HrishiKabra/reefscan.git
%cd reefscan
!pip install -q torch==2.4.1 torchvision==0.19.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q transformers==4.44.2 safetensors==0.4.5 huggingface_hub==0.25.2 \
                pyarrow==17.0.0 scikit-learn==1.5.2 pillow==10.4.0 matplotlib==3.9.2

> If the next cell throws a NumPy/ABI error, do **Runtime -> Restart session**, then re-run
> from the `%cd reefscan` line below (Colab ships a preinstalled NumPy; a restart clears it).

In [ ]:
%cd /content/reefscan
import torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0))

## 3. Rung 1 — PyTorch fp32 baseline (the control)

In [ ]:
!PYTHONPATH=. python -m edge.run_baseline

## 4. Rung 2 — torch.compile (Inductor, max-autotune)
Same weights, same test set — only the execution graph changes (op fusion + autotuned kernels +
CUDA graphs). **The first run compiles per input shape and can take a few minutes** — that
one-time cost is reported separately and is NOT included in the benchmarked latency.

If `max-autotune` errors on this stack, set the mode and re-run this cell:
`%env REEFSCAN_COMPILE_MODE=reduce-overhead`

In [ ]:
!PYTHONPATH=. python -m edge.run_rung2

## 5. The results table (CPU local + GPU rungs)

In [ ]:
print(open("edge/RESULTS.md").read())

## 6. Send the numbers back
Copy this cell's output and paste it back into the chat — it gets committed. (Or use the
download cell below.)

In [ ]:
print(open("edge/results.csv").read())

In [ ]:
# optional: download the CSV instead of copy-pasting
from google.colab import files
files.download("edge/results.csv")